 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison Mohamed Abdelnasser Gomaa Mohamed
Name           :                 <br>
Student Number : L XXXXX         <br>
Due Date       :                 <br>
Assignment     : CA2             <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

## Imports

In [2]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import pytesseract

import fitz  

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import spacy

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# download/load NLP 

In [3]:
nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

[nltk_data] Downloading package punkt to /Users/apple/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /Users/apple/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Support Functions

In [27]:
def get_category(filename):
    return filename.split("_")[0].lower()

In [28]:
def extract_text_image(file_path):
    image = Image.open(file_path)
    text = pytesseract.image_to_string(image)
    return text

In [29]:
def extract_text_pdf(file_path):
    doc = fitz.open(file_path)
    full_text = ""

    for page_number, page in enumerate(doc):
        text = page.get_text()

        if text.strip():
            full_text += text + "\n"
        else:
            pix = page.get_pixmap(dpi=300)
            image_path = f"temp_page_{page_number}.png"
            pix.save(image_path)

            image = Image.open(image_path)
            ocr_text = pytesseract.image_to_string(image)
            full_text += ocr_text + "\n"

    return full_text

# NLP

## Sub Heading 1

In [5]:
data_folder = Path("data/Datasets")

files = list(data_folder.glob("*"))

files

[PosixPath('data/Datasets/reports.pdf'),
 PosixPath('data/Datasets/form_02.tif'),
 PosixPath('data/Datasets/form_03.tif'),
 PosixPath('data/Datasets/form_01.tif'),
 PosixPath('data/Datasets/advertisement_03.tif'),
 PosixPath('data/Datasets/invoices.pdf'),
 PosixPath('data/Datasets/advertisement_02.tif'),
 PosixPath('data/Datasets/emails.pdf'),
 PosixPath('data/Datasets/advertisement_01.tif')]

In [15]:
records = []

for file in files:
    suffix = file.suffix.lower()

    if suffix in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]:
        raw_text = extract_text_image(file)

    elif suffix == ".pdf":
        raw_text = extract_text_pdf(file)

    else:
        continue

    records.append({
        "filename": file.name,
        "category": get_category(file.stem),
        "file_type": suffix,
        "raw_text": raw_text
    })

df = pd.DataFrame(records)

df.head()

,filename,category,file_type,raw_text
0,reports.pdf,reports,.pdf,Se\nTable 1. Magnitudes of the Association bet...
1,form_02.tif,form,.tif,"REGION # 4,614, 24 DATE: __ 6/1198\nTotal numb..."
2,form_03.tif,form,.tif,To: KASPARROW _ DATE TONYO:\nFROM: H Alcala”\n...
3,form_01.tif,form,.tif,HARLEY.\n\n———_\nSCHEDULE #: 94-089 Dave: 4/20...
4,advertisement_03.tif,advertisement,.tif,SURGEON GENERAL'S WARNING: Cigarette\nSmoke Co...


In [16]:
for i in range(len(df)):
    print("=" * 80)
    print("File:", df.loc[i, "filename"])
    print("Category:", df.loc[i, "category"])
    print("-" * 80)
    print(df.loc[i, "raw_text"][:1000])

File: reports.pdf
Category: reports
--------------------------------------------------------------------------------
Se
Table 1. Magnitudes of the Association between Lung Cancer and Selected Risk

Factors

ooo

Risk Factors Relative Risk Reference

eee
Fruit Consumption:

High vs low intake 0.13 Knekt et al. 1991
Finnish nonsmoking males (p for trend
vs population controls <0.001)
High vs low intake 0.26 Fraser et al. 199]
U.S. Seventh-day Adventists (0.10-0.70)
cases vs controls
High vs low intake 0.4 Koo 1988
Asian female cases vs controls (0.2-0.9)
Dietary Carotene
Medium vs low intake of 0.2 Pastorino et al.
dietary beta-carotene (p <0.05) 1987
Female cases vs population controls
High vs low intake 0.3 Byers et al. 1987
of carotene from fruits (p for trend
and vegetables = 0.06)

Never and ex-smoking female
cases vs population controls

High vs. low intake of 0.3 Le Marchand et al.
other dietary carotenoids (pfortrend 1989
with vitamin A activity = 0.009)

Female cases vs populati

In [31]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["raw_text"].apply(clean_text)

df[["filename", "clean_text"]].head()

,filename,clean_text
0,reports.pdf,se table 1 magnitudes of the association betwe...
1,form_02.tif,region 4 614 24 date 6 1198 total number of ch...
2,form_03.tif,to kasparrow date tonyo from h alcala div name...
3,form_01.tif,harley schedule 94 089 dave 4 20 04 program ar...
4,advertisement_03.tif,surgeon general s warning cigarette smoke cont...


In [18]:
def tokenize_text(text):
    return nltk.word_tokenize(text)

df["tokens"] = df["clean_text"].apply(tokenize_text)

df[["filename", "tokens"]].head()

,filename,tokens
0,reports.pdf,"[se, table, 1, magnitudes, of, the, associatio..."
1,form_02.tif,"[region, 4, 614, 24, date, 6, 1198, total, num..."
2,form_03.tif,"[to, kasparrow, date, tonyo, from, h, alcala, ..."
3,form_01.tif,"[harley, schedule, 94, 089, dave, 4, 20, 04, p..."
4,advertisement_03.tif,"[surgeon, general, s, warning, cigarette, smok..."


In [20]:
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words and len(word) > 2]

df["tokens_no_stopwords"] = df["tokens"].apply(remove_stopwords)

df[["filename", "tokens_no_stopwords"]].head()

,filename,tokens_no_stopwords
0,reports.pdf,"[table, magnitudes, association, lung, cancer,..."
1,form_02.tif,"[region, 614, date, 1198, total, number, chain..."
2,form_03.tif,"[kasparrow, date, tonyo, alcala, div, name, ra..."
3,form_01.tif,"[harley, schedule, 089, dave, program, area, t..."
4,advertisement_03.tif,"[surgeon, general, warning, cigarette, smoke, ..."


In [22]:
def stem_tokens(tokens):
    return [stemmer.stem(word) for word in tokens]

df["stemmed_tokens"] = df["tokens_no_stopwords"].apply(stem_tokens)

df[["filename", "stemmed_tokens"]].head()

,filename,stemmed_tokens
0,reports.pdf,"[tabl, magnitud, associ, lung, cancer, select,..."
1,form_02.tif,"[region, 614, date, 1198, total, number, chain..."
2,form_03.tif,"[kasparrow, date, tonyo, alcala, div, name, ra..."
3,form_01.tif,"[harley, schedul, 089, dave, program, area, ti..."
4,advertisement_03.tif,"[surgeon, gener, warn, cigarett, smoke, contai..."


In [23]:
def lemmatise_text(text):
    doc = nlp(text)
    lemmas = [
        token.lemma_
        for token in doc
        if token.text not in stop_words and token.is_alpha
    ]
    return lemmas

df["lemmas"] = df["clean_text"].apply(lemmatise_text)

df[["filename", "lemmas"]].head()

,filename,lemmas
0,reports.pdf,"[se, table, magnitude, association, lung, canc..."
1,form_02.tif,"[region, date, total, number, chain, headquart..."
2,form_03.tif,"[kasparrow, date, tonyo, h, alcala, div, name,..."
3,form_01.tif,"[harley, schedule, dave, program, area, timing..."
4,advertisement_03.tif,"[surgeon, general, warn, cigarette, smoke, con..."


In [24]:
def extract_entities(text):
    doc = nlp(text)
    entities = []

    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities

df["entities"] = df["raw_text"].apply(extract_entities)

df[["filename", "entities"]].head()

,filename,entities
0,reports.pdf,"[(1, CARDINAL), (Association, ORG), (Lung Canc..."
1,form_02.tif,"[(4,614, MONEY), (24, CARDINAL), (Chain Headqu..."
2,form_03.tif,"[(KASPARROW, ORG), (Sect IZ, PERSON), (1, CARD..."
3,form_01.tif,"[(HARLEY, ORG), (94-089, CARDINAL), (4/20/04, ..."
4,advertisement_03.tif,"[(Carbon Monoxide, PERSON)]"


# Vision

## Sub Heading 1

In [ ]:
# code here...

# Multi-modal

## Sub Heading 1

In [ ]:
# code here

# Final Output

In [ ]:
# code